# Sesión 6 · Agrupamiento de filas con Pandas

**Introducción a la Ingeniería en Ciencia de Datos — período 2026-3C**
Universidad del Magdalena · Facultad de Ingeniería

- Ing. Alexander Bustamante Martínez, MSc., PhD. — abustamante@unimagdalena.edu.co
- Ing. Andrés Eduardo Paternina Ariza, Esp. — apaternina@unimagdalena.edu.co

---

### ¿Qué vas a aprender hoy?

Al terminar este notebook serás capaz de:

1. Explicar qué es un **agrupamiento** y por qué todo `groupby` sigue tres pasos: **dividir → aplicar → combinar**.
2. Usar las **funciones de agregación** (`size`, `count`, `sum`, `mean`, `min`, `max`, `nunique`) y distinguir `size` de `count`.
3. Escribir la instrucción completa `df.groupby(...).agg(...).reset_index()` y leerla paso a paso.
4. **Filtrar y ordenar** el resultado de un agrupamiento.
5. Pedir **varias medidas** a la vez y agrupar por **dos columnas**.
6. Leer los resultados con **criterio**: cuántas filas respaldan cada grupo.

> **Cómo usar este notebook:** léelo de arriba hacia abajo y ejecuta cada celda de código en orden con `Shift + Enter`. En cada pregunta, escribe tu solución en la celda vacía siguiendo la explicación de clase.

---
## 1. Preparar los datos

Trabajamos con el mismo archivo de la sesión 5: `futbolistas_promesas_bananeras_2022_100.csv` (club ficticio **Promesas Bananeras**).

1. Descarga el archivo del **campus virtual**.
2. En Colab, súbelo a `/content` (panel izquierdo → ícono de **carpeta 📁 → Subir**).

> ⚠️ Los archivos subidos a `/content` se **borran** al cerrar la sesión de Colab. Si vuelves otro día, súbelo de nuevo.

In [9]:
import pandas as pd
import numpy as np

# Ruta del archivo en Colab. Si trabajas en tu computador (Jupyter local),
# cambia la ruta por "futbolistas_promesas_bananeras_2022_100.csv".
RUTA = "/content/futbolistas_promesas_bananeras_2022_100.csv"

df = pd.read_csv(RUTA)
df.head()

,player_id,full_name,nationality,position,dominant_foot,birth_date,fitness_level,joined_year,left_year,matches_played,goals,assists,yellow_cards,red_cards,height_cm,weight_kg,salary_cop_million_month
0,PB2022001,Santiago Rodríguez,Ecuador,Mediapunta,Derecha,2000-12-04,Medio,2024,2024.0,18,4,4,3,0,177.3,81.2,6.36
1,PB2022002,Wilson González,España,Lateral Derecho,Derecha,2005-01-25,Medio,2023,2024.0,25,0,1,2,0,165.3,71.0,7.04
2,PB2022003,Jhon Suárez,Ecuador,Extremo Derecho,Derecha,2007-09-04,Élite,2022,NaN,108,9,18,17,0,179.0,60.3,9.36
3,PB2022004,Nicolás Jiménez,Nigeria,Mediocentro Defensivo,Derecha,2001-02-13,Medio,2022,NaN,77,4,11,13,0,180.9,83.1,1.41
4,PB2022005,Diego Márquez,Chile,Extremo Derecho,Ambas,1998-03-15,Alto,2023,2023.0,27,2,11,4,0,178.1,71.7,1.42


### 1.1 La limpieza de la sesión 5, en una sola celda

Antes de agrupar, dejamos la tabla como la dejamos la clase pasada: **primero limpieza, después análisis**.

> ⚠️ **¿Por qué esta vez no convertimos `nationality`, `position` y `dominant_foot` a `category`?**
> Cuando se agrupa por columnas categóricas, algunas versiones de Pandas muestran **también las combinaciones que no existen** (con conteo 0). Para evitarlo, esas tres columnas quedan como **texto limpio**.

In [10]:
# 1. Texto → fecha
df["birth_date"] = pd.to_datetime(df["birth_date"], errors="coerce")

# 2. Años como enteros que admiten vacíos (Int64, con I mayúscula)
df["left_year"] = pd.to_numeric(df["left_year"], errors="coerce").astype("Int64")
df["joined_year"] = df["joined_year"].astype("Int64")

# 3. Texto consistente: sin espacios sobrantes y con mayúscula inicial
df["nationality"] = df["nationality"].astype("category")
df["position"] = df["position"].astype("category")
df["dominant_foot"] = df["dominant_foot"].astype("category")

# 4. Variable ordinal: Bajo < Medio < Alto < Élite
fitness_order = ["Bajo", "Medio", "Alto", "Élite"]
df["fitness_level"] = pd.Categorical(df["fitness_level"],
                                     categories=fitness_order,
                                     ordered=True)

print("Dimensiones:", df.shape)

Dimensiones: (100, 17)


### 1.2 Diccionario de datos

**Quién es el jugador**

| Variable | Qué registra | Ejemplo |
|---|---|---|
| `player_id` | Código único del jugador; no se repite | PB2022001 |
| `full_name` | Nombre completo | Santiago Rodríguez |
| `nationality` | País de origen (16 países) | Ecuador |
| `position` | Posición en la cancha (10 posiciones) | Mediapunta |
| `dominant_foot` | Pierna con la que mejor juega | Derecha, Izquierda o Ambas |
| `birth_date` | Fecha de nacimiento; **llega como texto** | 2000-12-04 |
| `fitness_level` | Condición física; **sus niveles tienen orden** | Bajo < Medio < Alto < Élite |
| `height_cm` | Estatura en centímetros | 177.3 |
| `weight_kg` | Peso en kilogramos | 81.2 |

**Su paso por el club**

| Variable | Qué registra | Ejemplo |
|---|---|---|
| `joined_year` | Año en que llegó al club (2022 a 2025) | 2024 |
| `left_year` | Año en que se fue; **vacío si sigue en el club** | 2024 o vacío |
| `matches_played` | Partidos jugados | 18 |
| `goals` | Goles anotados | 4 |
| `assists` | Asistencias (pases de gol) | 4 |
| `yellow_cards` | Tarjetas amarillas recibidas | 3 |
| `red_cards` | Tarjetas rojas recibidas | 0 |
| `salary_cop_million_month` | Salario mensual, en millones de pesos (COP) | 6.36 = $6,36 millones |

**Detalles del archivo que cambian el análisis**

1. **Un vacío en `left_year` no es un error:** el jugador sigue en el club.
2. **`birth_date` llega como texto:** se convierte con `pd.to_datetime()` antes de calcular edades.
3. **`fitness_level` tiene un orden:** ordenar alfabéticamente daría un resultado falso.
4. **La unidad está en el nombre de la columna:** `height_cm` → 1,80 m = 180 cm; `salary_cop_million_month` → millones de pesos al mes.

### ❓ ¿Cuáles variables son cuantitativas y cuáles cualitativas, y a qué categoría dentro de estas pertenecen?

| Tipo | Columnas |
|---|---|
| Cuantitativa discreta | |
| Cuantitativa continua | |
| Cualitativa nominal | |
| Cualitativa ordinal | |

### 1.3 Tres confusiones frecuentes al usar Pandas

| Confusión | Lo correcto | Para recordar |
|---|---|---|
| ¿Simples o dobles? | `df["goals"]` → Series · `df[["goals"]]` → DataFrame | Varias columnas siempre van con corchetes dobles. |
| Una tabla nueva | `g = df.sort_values("goals")` ✓ | Pasa igual con `dropna()`, `set_index()`, `assign()`… **y `groupby().agg()`**. |
| Filtrar bien | `==` compara, `=` asigna · `(c1) & (c2)` · `.isna()` | Al filtrar, revisa cuántas filas quedaron con `.shape`. |

---
## 2. Agrupamiento: la idea

**Agrupamiento (`groupby`):** segmenta los datos en **grupos** a partir de valores comunes de una columna, y luego **resume** cada grupo.



### 2.1 Los tres pasos: dividir, aplicar, combinar

| Paso | Qué hace |
|---|---|
| 1. **DIVIDIR** | Separa las filas en grupos |
| 2. **APLICAR** | Resume cada grupo por separado |
| 3. **COMBINAR** | Une los resúmenes en una tabla |

> 📌 **Para recordar:** después de agrupar, **cada fila del resultado es un grupo, no un jugador**. El resultado tiene tantas filas como grupos distintos.

### 2.2 Cómo se lee una instrucción de agrupamiento

```python
df.groupby("nationality")                   # 1. DIVIDIR  → ¿por qué columna se forman los grupos?
  .agg(cantidad = ("player_id", "size"))    # 2. APLICAR  → nombre_nuevo = ("columna", "función")
  .reset_index()                            # 3. COMBINAR → devuelve una tabla ordinaria
```

1. **`df.groupby("nationality")`** aún **no calcula nada**: solo separa las filas.
2. **`.agg(nombre_nuevo = ("columna", "función"))`**: qué columna se resume, con qué función y cómo se llamará el resultado.
3. **`.reset_index()`**: la columna de grupos deja de ser índice y vuelve a ser columna.

> 🛠️ **Tip del ingeniero:** antes de escribir, responde tres cosas: **¿agrupar por qué? ¿resumir qué columna? ¿con qué función?**

### 2.3 Funciones de agregación

| Función | Qué calcula en cada grupo | Ejemplo con el club |
|---|---|---|
| `size` | Cuántas filas tiene el grupo (**con** vacíos) | Jugadores por nacionalidad |
| `count` | Cuántos valores **no vacíos** hay | Retirados por nacionalidad |
| `sum` | La suma de los valores | Goles por posición |
| `mean` | El promedio | Salario promedio por posición |
| `min` / `max` | El menor / el mayor valor | Estatura por pierna dominante |
| `nunique` | Cuántos valores distintos hay | Posiciones distintas por país |

> ⚠️ **`size` y `count` parecen iguales, pero no lo son cuando hay vacíos.**
>
> ⚠️ **Sé explícito:** indica siempre qué columna resumir. Sumar todas las columnas produce cifras sin sentido, como la suma de `joined_year`.
>
> 🛠️ Usa `size` para saber **cuántos son**. Usa `count` para saber **cuántos tienen datos**.

---
## 3. Preguntas de clase

Para cada pregunta, antes de escribir código responde: **¿por qué columna divido? → ¿qué resumo y con qué función? → ¿qué hago después (filtrar, ordenar)?**

### ❓ Pregunta 1: ¿Cuál es la distribución de jugadores por nacionalidad (de mayor a menor)?

In [25]:
# Escribe aquí tu solución a la Pregunta 1

### ❓ Pregunta 2: ¿Cuál es la cantidad de goles que ha aportado cada posición?

In [ ]:
# Escribe aquí tu solución a la Pregunta 2

### ❓ Pregunta 3: ¿Cuál es la cantidad de goles por cada posición, conservando únicamente aquellas que superen los 40 goles?

*Ordenando de mayor a menor.*

In [ ]:
# Escribe aquí tu solución a la Pregunta 3

### ❓ Pregunta 4: ¿Cuál es el rango de estatura (mínimo y máximo) asociado a cada pierna dominante?

In [ ]:
# Escribe aquí tu solución a la Pregunta 4

### ❓ Pregunta 5: ¿Cuál es la cantidad de asistencias que se registra por cada pierna dominante según la nacionalidad?

*Ordenados alfabéticamente por la nacionalidad y, dentro de la misma nacionalidad, por la cantidad de goles.*

In [ ]:
# Escribe aquí tu solución a la Pregunta 5

---
## 4. Antes de escribir `groupby`, responde tres preguntas

| Pregunta | Pista | Código |
|---|---|---|
| 1. ¿Por qué columna divido? | La palabra «**por cada…**» en la pregunta la indica. | `groupby("position")` |
| 2. ¿Qué resumo y con qué función? | Nombra la columna: sin ella, Pandas resume todas las numéricas. | `("goals", "sum")` |
| 3. ¿Qué hago con el resultado? | Filtrar, ordenar o calcular: es un DataFrame como cualquier otro. | `.sort_values(...)` |

> ✋ **Detente y piensa:** un promedio calculado sobre un grupo de 1 o 2 jugadores puede cambiar con un solo dato. Antes de concluir, **mira cuántas filas tiene cada grupo**.

> 💬 **Pregunta de control:** «¿qué decisión cambia gracias a esta tabla?». Si no sabes responderla, vuelve a formular la pregunta antes de seguir escribiendo código.

---
## 5. Taller en grupos

Reúnanse en **grupos de tres** y resuelvan las siguientes necesidades de datos. Escriban su solución en la celda vacía de cada ejercicio.

### 🧩 Ejercicio 1

**¿Cuál es el total acumulado de tarjetas rojas y amarillas por cada nacionalidad?**
Ordena el resultado priorizando a las nacionalidades con **más tarjetas rojas** (descendente).

In [ ]:
# Escribe aquí tu solución al Ejercicio 1

### 🧩 Ejercicio 2

**¿Cuál es el peso promedio y la altura máxima por posición, pero muestra únicamente aquellas cuyo peso promedio supere los 75 kg?**
Ordenadas de mayor a menor peso.

In [ ]:
# Escribe aquí tu solución al Ejercicio 2

### 🧩 Ejercicio 3

**¿Cuál es el promedio de partidos jugados y el salario máximo por nacionalidad, pero solo para aquellos países que tengan o hayan tenido más de 5 jugadores en la plantilla?**

In [ ]:
# Escribe aquí tu solución al Ejercicio 3

### 🧩 Ejercicio 4

**¿Cuál es el salario promedio agrupado por posición y pie dominante?**
Ordena el reporte **alfabéticamente por posición**, pero dentro de cada posición muestra primero el pie dominante **mejor pagado**.

In [ ]:
# Escribe aquí tu solución al Ejercicio 4

### 🧩 Ejercicio 5

**¿Qué combinaciones de posición y pie dominante tienen un promedio de más de 7 tarjetas amarillas por jugador?**
Muestra primero las que más tarjetas tienen.

In [ ]:
 # Escribe aquí tu solución al Ejercicio 5